# Scaling Law for the Optimal Detuning of a Transport-Mediated Gate

**Self-contained** — run top to bottom, no dependency on the other notebooks.

## The analytic argument (derive this properly in the report)

In the dispersive regime the effective exchange coupling is $J(t) = g(t)^2/\Delta$,
so the accumulated exchange angle is $\theta = \int J\,dt$. Fixing $\theta$ fixes
the gate (full exchange = iSWAP).

**Cavity-loss penalty.** The virtual photon population is $n(t) \approx (g(t)/\Delta)^2$,
so the total photon loss is

$$\kappa \int n\,dt = \frac{\kappa}{\Delta^2}\int g^2 dt = \frac{\kappa\theta}{\Delta}.$$

Note this is **independent of the pulse shape** at fixed $\theta$ — it depends only on
the detuning. (This is a result in its own right: shaping the transit profile can only
help through the *duration*, not through the cavity-loss channel.)

**Atomic-decay penalty.** The gate time at fixed $\theta$ scales as
$T \propto \theta\Delta/g_0^2$, so the decay cost is $\gamma T \propto \gamma\theta\Delta/g_0^2$.

**Total infidelity and the optimum.**

$$1 - F \;\approx\; A\,\gamma\Delta \;+\; B\,\frac{\kappa}{\Delta}
\qquad\Longrightarrow\qquad
\boxed{\;\Delta_{\rm opt} \propto g_0\sqrt{\kappa/\gamma}\;}$$

**A square-root scaling law.** This notebook tests it, and identifies where it fails.

*Expected outcome:* the law should hold when $\Delta_{\rm opt} \gg g_0$ (genuinely
dispersive) and break down when $\Delta_{\rm opt} \sim g_0$ (the resonant–dispersive
crossover) — which is exactly where the earlier four-point data sat, explaining why
no clean law appeared there.

In [ ]:
using QuantumOptics
using LinearAlgebra
using Plots
gr()

const n_max = 2
const g0    = 1.0
const w     = 1.0

b_cav = FockBasis(n_max)
b_at  = SpinBasis(1//2)
Ic, Ia = one(b_cav), one(b_at)

a   = destroy(b_cav) ⊗ Ia ⊗ Ia
s1m = Ic ⊗ sigmam(b_at) ⊗ Ia
s2m = Ic ⊗ Ia ⊗ sigmam(b_at)
sz1 = Ic ⊗ sigmaz(b_at) ⊗ Ia
sz2 = Ic ⊗ Ia ⊗ sigmaz(b_at)

num  = dagger(a) * a
exc1 = (sz1 + one(sz1)) / 2
exc2 = (sz2 + one(sz2)) / 2
Hc1  = dagger(a) * s1m + a * dagger(s1m)
Hc2  = dagger(a) * s2m + a * dagger(s2m)

gpulse(t, t0, v) = g0 * exp(-((v * (t - t0)) / w)^2)

labels      = ["gg", "ge", "eg", "ee"]
ket_at(c)   = c == 'e' ? spinup(b_at) : spindown(b_at)
full_ket(s) = fockstate(b_cav, 0) ⊗ ket_at(s[1]) ⊗ ket_at(s[2])

println("setup ok")

## Fidelity at fixed exchange angle

The transit velocity is recalibrated at every $\Delta$ as $v \propto 1/\Delta$, which
holds $\theta$ **fixed** as $\Delta$ varies. This is what makes the scaling argument
applicable — otherwise you would be changing the gate as well as the detuning.

In [ ]:
function gate_fid(Δ, κv, γv; nsteps = 1200)
    v  = g0^2 * w * sqrt(2/pi) / Δ          # fixed θ  =>  v ∝ 1/Δ
    t0 = 4*(w/v)
    T  = range(0, 8*(w/v), length = nsteps)
    H0 = Δ*(exc1 + exc2)
    Jl = AbstractOperator[]
    κv > 0 && push!(Jl, sqrt(κv)*a)
    γv > 0 && push!(Jl, sqrt(γv)*s1m)
    γv > 0 && push!(Jl, sqrt(γv)*s2m)
    Jd = dagger.(Jl)
    f(t, ρ) = (H0 + gpulse(t, t0, v)*(Hc1 + Hc2), Jl, Jd)
    ideal = ["gg", "eg", "ge", "ee"]        # iSWAP action, in populations
    F = 0.0
    for (j, si) in enumerate(labels)
        _, ρt = timeevolution.master_dynamic(T, full_ket(si), f)
        ψid = full_ket(ideal[j])
        F += real(dagger(ψid) * (ρt[end] * ψid))
    end
    return F/4
end

println("gate_fid ready — quick check:")
println("  Δ=4, κ=0.1, γ=0.002 -> F = ", round(gate_fid(4.0, 0.1, 0.002), digits=4))

## Locating $\Delta_{\rm opt}$

Coarse scan, then parabolic refinement about the peak. Boundary maxima are **rejected**:
a peak at the edge of the sweep window is not a real optimum (this was the flaw in the
earlier $\kappa=0$ point). If everything reports `interior = false`, widen `Δhi`.

In [ ]:
function find_Δopt(κv, γv; Δlo = 1.5, Δhi = 30.0, n = 18, verbose = false)
    Δs = range(Δlo, Δhi, length = n)
    Fs = [gate_fid(Δ, κv, γv) for Δ in Δs]
    i  = argmax(Fs)
    verbose && println("   scan max at Δ=", round(Δs[i],digits=2), " F=", round(Fs[i],digits=4))
    (i == 1 || i == n) && return (Δs[i], Fs[i], false)   # boundary -> not a real optimum
    x1,x2,x3 = Δs[i-1], Δs[i], Δs[i+1]
    y1,y2,y3 = Fs[i-1], Fs[i], Fs[i+1]
    den  = (y1 - 2y2 + y3)
    Δopt = den == 0 ? x2 : x2 - 0.5*(x3-x1)*(y3-y1)/(4*den)
    return (Δopt, y2, true)
end

println("find_Δopt ready")

## The sweep

Vary $\kappa/\gamma$ over roughly two decades at fixed small $\gamma$. Small $\gamma$
pushes $\Delta_{\rm opt}$ out into the genuinely dispersive regime where the prediction
should hold.

**This cell is the slow one** — gate time grows as $\Delta$, so the large-$\Delta$ points
take longest. Start with the short `κ_list` below to check it runs, then extend.

In [ ]:
γ_fixed = 0.002
κ_list  = [0.02, 0.05, 0.1, 0.2, 0.4, 0.8, 1.6]
# to test quickly first, use:  κ_list = [0.05, 0.2, 0.8]

ratios, Δopts, Fopts, interior = Float64[], Float64[], Float64[], Bool[]
println("γ = ", γ_fixed)
println(" κ        κ/γ       Δ_opt      F_opt     interior?")
for κv in κ_list
    Δo, Fo, ok = find_Δopt(κv, γ_fixed)
    push!(ratios, κv/γ_fixed); push!(Δopts, Δo)
    push!(Fopts, Fo); push!(interior, ok)
    println(rpad(κv,9), rpad(round(κv/γ_fixed,digits=1),10),
            rpad(round(Δo,digits=3),11), rpad(round(Fo,digits=4),10), ok)
end

## Fit the exponent

$\Delta_{\rm opt} \propto (\kappa/\gamma)^p$, prediction $p = 0.5$. Only interior
optima are fitted.

In [ ]:
sel = interior
if sum(sel) < 3
    println("Not enough interior optima to fit — widen Δhi and rerun.")
else
    x = log.(ratios[sel]); y = log.(Δopts[sel]); n = length(x)
    p = (n*sum(x.*y) - sum(x)*sum(y)) / (n*sum(x.^2) - sum(x)^2)
    c = (sum(y) - p*sum(x))/n
    R2 = 1 - sum((y .- (p.*x .+ c)).^2)/sum((y .- sum(y)/n).^2)

    println("fitted exponent p = ", round(p, digits=3), "   (prediction: 0.5)")
    println("R² = ", round(R2, digits=4))
    println(abs(p-0.5) < 0.08 ?
        "=> consistent with sqrt(κ/γ) scaling" :
        "=> deviates — check whether Δ_opt is still near g0 (crossover regime)")

    plot(ratios[sel], Δopts[sel], seriestype=:scatter, m=:circle, ms=6,
         xscale=:log10, yscale=:log10, label="numerics",
         xlabel="κ / γ", ylabel="Δ_opt / g₀",
         title="Scaling of the optimal detuning")
    plot!(ratios[sel], exp(c).*ratios[sel].^p, lw=2, label="fit: p = $(round(p,digits=3))")
    plot!(ratios[sel], exp(c).*ratios[sel].^0.5, ls=:dash, lw=2, label="prediction: p = 0.5")
end

---
# Extension: gate between atoms in **different** cavities

The natural use of the coupled-cavity (JCH) machinery. Two cavities connected by photon
hopping $J_{\rm hop}$, with **one atom transported through each**:

$$\hat H = \Delta(\hat n_{e1}+\hat n_{e2})
+ J_{\rm hop}(\hat a_1^\dagger \hat a_2 + \text{h.c.})
+ g_1(t)(\hat a_1^\dagger\hat\sigma_1^- + \text{h.c.})
+ g_2(t)(\hat a_2^\dagger\hat\sigma_2^- + \text{h.c.})$$

The atoms never share a cavity — they interact only through a photon that hops between
cavities. In the dispersive limit this gives an effective coupling
$J_{\rm eff} \sim g_0^2 J_{\rm hop}/\Delta^2$, i.e. a **photon-mediated interaction
between spatially separated atoms**.

**The question worth asking:** how does gate fidelity depend on the inter-cavity hopping
$J_{\rm hop}$ — and, extended to a chain, on the *separation* between the atoms? That is
a range-versus-fidelity trade-off which the single-cavity model cannot pose, and it uses
the lattice code directly.

*Untested starter code — treat as a scaffold, not a finished result.*

In [ ]:
# --- two coupled cavities, one atom in each -------------------------
bc = FockBasis(1)                       # single-excitation physics: cutoff 1 is enough
ba = SpinBasis(1//2)
Ic2, Ia2 = one(bc), one(ba)

A1 = destroy(bc) ⊗ Ic2 ⊗ Ia2 ⊗ Ia2      # cavity 1
A2 = Ic2 ⊗ destroy(bc) ⊗ Ia2 ⊗ Ia2      # cavity 2
S1 = Ic2 ⊗ Ic2 ⊗ sigmam(ba) ⊗ Ia2       # atom 1 (in cavity 1)
S2 = Ic2 ⊗ Ic2 ⊗ Ia2 ⊗ sigmam(ba)       # atom 2 (in cavity 2)
Z1 = Ic2 ⊗ Ic2 ⊗ sigmaz(ba) ⊗ Ia2
Z2 = Ic2 ⊗ Ic2 ⊗ Ia2 ⊗ sigmaz(ba)

E1 = (Z1 + one(Z1))/2
E2 = (Z2 + one(Z2))/2
Hop = dagger(A1)*A2 + dagger(A2)*A1              # photon hopping
C1  = dagger(A1)*S1 + A1*dagger(S1)              # atom1 <-> cavity1
C2  = dagger(A2)*S2 + A2*dagger(S2)              # atom2 <-> cavity2
Ntot = dagger(A1)*A1 + dagger(A2)*A2

vac2 = fockstate(bc,0) ⊗ fockstate(bc,0)
kt(c) = c == 'e' ? spinup(ba) : spindown(ba)
fk2(s) = vac2 ⊗ kt(s[1]) ⊗ kt(s[2])

function run_two_cav(Δ, Jhop; v = 0.05, κv = 0.0, γv = 0.0,
                     at = "eg", nsteps = 2000)
    t0 = 4*(w/v); T = range(0, 8*(w/v), length = nsteps)
    H0 = Δ*(E1 + E2) + Jhop*Hop
    Jl = AbstractOperator[]
    κv > 0 && (push!(Jl, sqrt(κv)*A1); push!(Jl, sqrt(κv)*A2))
    γv > 0 && (push!(Jl, sqrt(γv)*S1); push!(Jl, sqrt(γv)*S2))
    f(t, ρ) = (H0 + gpulse(t, t0, v)*(C1 + C2), Jl, dagger.(Jl))
    tout, ρt = timeevolution.master_dynamic(T, fk2(at), f)
    return tout, ρt
end

# does excitation move from atom 1 to atom 2 through the hopping photon?
tt, ρρ = run_two_cav(8.0, 1.0; v = 0.02)
p1 = real.(expect(E1, ρρ)); p2 = real.(expect(E2, ρρ))
nn = real.(expect(Ntot, ρρ))
println("max transfer to atom 2 = ", round(maximum(p2), digits=3))
println("peak total photon number = ", round(maximum(nn), digits=4))

plot(tt, p1, lw=2, label="atom 1 (cavity 1)", xlabel="time (1/g0)", ylabel="population",
     title="Atoms in separate cavities, coupled by photon hopping")
plot!(tt, p2, lw=2, label="atom 2 (cavity 2)")
plot!(tt, nn, lw=2, label="total photons (should stay small)")

**What to look for.** If excitation moves from atom 1 to atom 2 while the total
photon number stays small, you have a dispersive gate mediated across two cavities —
atoms that never share a mode, coupled by a virtual hopping photon.

**If nothing moves,** the effective coupling $\sim g_0^2 J_{\rm hop}/\Delta^2$ is much
weaker than the single-cavity case, so the transit must be correspondingly slower: reduce
`v` (try `v = 0.005`) or increase `Jhop`.

**The result to chase, if this works:** sweep $J_{\rm hop}$ and extract the effective
coupling or the gate fidelity, then check it against the predicted $g_0^2J_{\rm hop}/\Delta^2$
scaling. That is the same "derive, predict, test" structure as the first half of this
notebook, applied to a genuinely spatial question.